In [ ]:

import streamlit as st
import pandas as pd
import duckdb
import torch
import base64
import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import boto3
import json
import time

# AWS Bedrock client setup
bedrock_client = boto3.client("bedrock", region_name="us-east-1")  # Change region as needed

# AWS Bedrock Model and Endpoint
bedrock_model_id = "amazon.titan-txt-embeddings-v2"  # Amazon Titan Text Embeddings v2 Model ID

# Load Excel Data
file_path = "Banking_Dataset.xlsx"
xls = pd.ExcelFile(file_path)
dfs = {}
for sheet in xls.sheet_names:
    df = xls.parse(sheet)
    df.columns = [col.strip().replace(" ", "_") for col in df.columns]
    dfs[sheet] = df

# Register in DuckDB
conn = duckdb.connect(database=":memory:")
for name, df in dfs.items():
    conn.register(name, df)

# Generate column descriptions
column_descriptions = []
table_column_lookup = []
for table_name, df in dfs.items():
    for col in df.columns:
        alias = f"{table_name}.{col}"
        column_descriptions.append(alias)
        table_column_lookup.append((table_name, col))

# Vectorize using TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(column_descriptions).toarray().astype('float32')

# Create FAISS index
dimension = tfidf_matrix.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(tfidf_matrix)

# Cache for storing SQL queries based on natural language questions
sql_cache = {}

# Store question and SQL histories
question_history = []
sql_history = []
question_embeddings = []
sql_embeddings = []

# Function to get the database schema
def get_database_schema():
    schema = {}
    for table_name in dfs:
        result = conn.execute(f"DESCRIBE {table_name}").fetchdf()
        schema[table_name] = ", ".join(result['column_name'].tolist())
    return schema

# Function to check SQL validity
def is_valid_sql(query):
    try:
        conn.execute(f"EXPLAIN {query}")
        return True
    except:
        return False

# Function to run SQL queries
def run_sql_query(query):
    try:
        result_df = conn.execute(query).fetchdf()
        if result_df.empty:
            return "No matching records found.", None
        return result_df.to_html(index=False, classes='table table-striped'), result_df
    except Exception as e:
        return f"Error executing query: {e}", None

# Function to generate SQL from natural language query using Bedrock API
def generate_sql_from_bedrock(natural_query):
    # Check cache first
    if natural_query in sql_cache:
        st.write("Using cached SQL query...")
        return sql_cache[natural_query]

    schema = get_database_schema()
    schema_text = "\n".join([f"{table}: {cols}" for table, cols in schema.items()])

    prompt = f"""
You are an AI SQL assistant. Convert the question below into a valid DuckDB SQL query.

Database Schema:
{schema_text}

Question: {natural_query}

SQL Query:
"""
    # Making API request to AWS Bedrock
    try:
        response = bedrock_client.invoke_model(
            modelId=bedrock_model_id,
            body=json.dumps({"input": prompt}),
            contentType="application/json",
            accept="application/json"
        )
        response_body = json.loads(response['body'].read().decode('utf-8'))
        sql_query = response_body.get('output', '').strip()

        if "SQL Query:" in sql_query:
            sql_query = sql_query.split("SQL Query:")[-1].strip()

        # Ensure the generated SQL is valid
        if any(word in sql_query.upper() for word in ["DROP", "DELETE", "--", "ALTER"]):
            return None

        if not is_valid_sql(sql_query):
            return None

        # Store user query and SQL
        question_history.append(natural_query)
        sql_history.append(sql_query)

        # Update FAISS index with new question and SQL embeddings
        question_emb = vectorizer.transform([natural_query]).toarray().astype('float32')
        sql_emb = vectorizer.transform([sql_query]).toarray().astype('float32')
        question_embeddings.append(question_emb[0])
        sql_embeddings.append(sql_emb[0])

        faiss_index.add(question_emb)
        faiss_index.add(sql_emb)

        # Cache the generated SQL for future use
        sql_cache[natural_query] = sql_query

        return sql_query

    except Exception as e:
        st.error(f"Error invoking Bedrock model: {e}")
        return None

# Function to search history for similar questions/SQL
def search_history_faiss(text, top_k=3):
    if not question_embeddings:
        return [], []

    query_vec = vectorizer.transform([text]).toarray().astype('float32')
    _, indices = faiss_index.search(query_vec, top_k)

    similar_questions = [question_history[i] for i in indices[0]]
    similar_sql_queries = [sql_history[i] for i in indices[0]]

    return similar_questions, similar_sql_queries

# Streamlit UI for user interaction
st.title("SQL Query Generator using AWS Bedrock")

# Input field for natural language query
user_query = st.text_input("Enter your natural language query:")

# Button to trigger query generation
if user_query:
    with st.spinner("Generating SQL query..."):
        # Generate SQL from Bedrock
        sql_query = generate_sql_from_bedrock(user_query)

        # Display results
        if sql_query:
            st.subheader("Generated SQL Query:")
            st.code(sql_query)

            # Run the generated SQL query
            result_html, result_df = run_sql_query(sql_query)
            st.subheader("Query Results:")
            st.write(result_html, unsafe_allow_html=True)
        else:
            st.error("Could not generate a valid SQL query.")

    # Search for similar questions from history using FAISS
    similar_questions, similar_sql_queries = search_history_faiss(user_query)

    if similar_questions:
        st.subheader("Similar Questions from History:")
        for i, question in enumerate(similar_questions):
            st.write(f"{i+1}. {question} - SQL: {similar_sql_queries[i]}")
    else:
        st.write("No similar questions found.")

# Query cache display for user convenience
st.sidebar.title("Query Cache")
st.sidebar.write("Frequently asked questions and their generated SQL queries:")
for question, sql_query in sql_cache.items():
    st.sidebar.write(f"Q: {question} - SQL: {sql_query}")